# Resident Evil Quality & Descriptive Checks

Simple QA pass over all CSVs
- File-level summary (rows, columns, file size)
- Missing values & duplicates
- Descriptive stats for numeric, boolean, and datetime columns
- Comment-quality checks: empty, very short, very long, non-text, duplicates, language distribution

In [1]:
import os
import glob
import pandas as pd
import numpy as np

In [ ]:
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)
pd.set_option('display.max_colwidth', 120)

REVIEWS_DIR = '../raw_reviews'
files = sorted(glob.glob(os.path.join(REVIEWS_DIR, '*.csv')))
files = files[1:]  # Exclude author_id_map.csv)
print(f'Found {len(files)} CSV file(s):')
for f in files:
    size_mb = os.path.getsize(f) / 1024 / 1024
    print(f'  {os.path.basename(f):60s} {size_mb:7.2f} MB')

## File-level summary

In [3]:
summary_rows = []
for f in files:
    df = pd.read_csv(f, low_memory=False)
    summary_rows.append({
        'file': os.path.basename(f),
        'rows': len(df),
        'cols': df.shape[1],
        'size_mb': round(os.path.getsize(f) / 1024 / 1024, 2),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df

,file,rows,cols,size_mb
0,resident_evil_2_remake.csv,80416,16,23.65
1,resident_evil_3_remake.csv,44799,16,18.45


## Per-file check

In [4]:
def check_file(path):
    name = os.path.basename(path)
    print('=' * 90)
    print(f'FILE: {name}')
    print('=' * 90)

    df = pd.read_csv(path, low_memory=False)
    print(f'Shape: {df.shape[0]:,} rows × {df.shape[1]} cols')
    print(f'Columns: {list(df.columns)}\n')

    # --- Dtypes ---
    print('--- Dtypes ---')
    print(df.dtypes)
    print()

    # --- Missing values ---
    print('--- Missing values per column ---')
    miss = df.isna().sum()
    miss_pct = (miss / len(df) * 100).round(2)
    miss_tbl = pd.DataFrame({'n_missing': miss, 'pct_missing': miss_pct})
    print(miss_tbl[miss_tbl['n_missing'] > 0] if (miss > 0).any() else 'No missing values.')
    print()

    # --- Duplicate rows ---
    n_dup = df.duplicated().sum()
    print(f'--- Duplicate rows: {n_dup:,} ({n_dup/len(df)*100:.2f}%) ---')
    if 'recommendationid' in df.columns:
        n_dup_id = df['recommendationid'].duplicated().sum()
        print(f'    Duplicate recommendationid values: {n_dup_id:,}')
    print()

    # --- Numeric describe ---
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if num_cols:
        print('--- Numeric describe ---')
        print(df[num_cols].describe().T.round(2))
        print()

    # --- Boolean / categorical describe ---
    bool_like = [c for c in df.columns if df[c].dropna().isin([True, False, 0, 1, 'True', 'False']).all() and df[c].nunique(dropna=True) <= 2]
    for c in bool_like:
        print(f'--- Value counts: {c} ---')
        print(df[c].value_counts(dropna=False))
        print()

    # --- Datetime / timestamp columns ---
    ts_cols = [c for c in df.columns if 'timestamp' in c.lower() or c.lower().endswith('_at') or c.lower() in ('created', 'updated')]
    for c in ts_cols:
        s = df[c].dropna()
        if s.empty:
            continue
        # Steam timestamps are unix seconds
        if pd.api.types.is_numeric_dtype(s):
            try:
                dt = pd.to_datetime(s, unit='s', errors='coerce')
                print(f'--- Time range ({c}, treated as unix seconds) ---')
                print(f'  min: {dt.min()}   max: {dt.max()}')
                print()
            except Exception:
                pass

    # --- Language distribution (if present) ---
    if 'language' in df.columns:
        print('--- Language distribution (top 15) ---')
        print(df['language'].value_counts(dropna=False).head(15))
        print()


    return df

In [5]:
# Run checks across every file
for f in files:
    _ = check_file(f)
    print()

FILE: resident_evil_2_remake.csv
Shape: 80,416 rows × 16 cols
Columns: ['review_id', 'author_id', 'author_number', 'review', 'review_length', 'sentiment', 'purchased', 'received_for_free', 'votes_up', 'votes_funny', 'date_created', 'date_updated', 'author_num_games_owned', 'author_num_reviews', 'author_playtime_forever_min', 'author_playtime_at_review_min']

--- Dtypes ---
review_id                            str
author_id                            str
author_number                        str
review                               str
review_length                      int64
sentiment                          int64
purchased                          int64
received_for_free                  int64
votes_up                           int64
votes_funny                        int64
date_created                         str
date_updated                         str
author_num_games_owned             int64
author_num_reviews                 int64
author_playtime_forever_min        int64
author_pl

## 3. Cross-file comparison

Quick side-by-side: rows, blanks, duplicate texts, average comment length.

In [14]:
rows = []
for f in files:
    df = pd.read_csv(f, low_memory=False)
    cc = None
    rec = {'file': os.path.basename(f), 'n_rows': len(df), 'n_dup_rows': int(df.duplicated().sum())}
    if cc is not None:
        text = df[cc].astype('string')
        lengths = text.fillna('').str.len()
        rec.update({
            'comment_col': cc,
            'n_blank_comments': int(text.fillna('').str.strip().eq('').sum()),
            'n_dup_comments': int(text.dropna().duplicated().sum()),
            'mean_chars': round(lengths.mean(), 1),
            'median_chars': float(lengths.median()),
            'p95_chars': float(lengths.quantile(0.95)),
        })
    rows.append(rec)

compare_df = pd.DataFrame(rows)
compare_df

,file,n_rows,n_dup_rows
0,resident_evil_0_339340.csv,7227,0
1,resident_evil_1996_4249100.csv,717,0
2,resident_evil_2_1998_4249110.csv,702,0
3,resident_evil_2_remake_883710.csv,76381,0
4,resident_evil_3_nemesis_1999_4249120.csv,787,0
5,resident_evil_3_remake_952060.csv,42427,0
6,resident_evil_4_2005_254700.csv,32825,0
7,resident_evil_4_remake_2050650.csv,89341,0
8,resident_evil_5_21690.csv,24886,0
9,resident_evil_6_221040.csv,21337,0
